# Week 5 — Retrieval 전략 Ablation

4주차 우승 청킹(B-large 1500/300)을 고정하고 **retriever만** 4가지로 바꿔 RAGAS 비교.

| 통제 변수 | 고정 값 |
|---|---|
| 데이터 | 소수몽키 5건 (`WEEK4_CONTENT_IDS`) |
| 청킹 | RecursiveCharacter 1500/300, baseline separators |
| 임베딩 | BAAI/bge-m3 |
| LLM | gpt-4o-mini (temp=0) |
| 평가셋 | `WEEK4_EVAL_DATA` 10문항 |
| RAGAS | Faithfulness/AnswerRelevancy/ContextPrecision/ContextRecall |
| 최종 k | 4 (4 strategy 공통) |

**바뀌는 것은 retriever 하나뿐.**

In [1]:
from advanced_retrieval.ingest import ingest
print('chunks ingested/confirmed:', ingest())

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

chunks ingested/confirmed: 68


In [2]:
from advanced_retrieval.retrievers import (
    build_dense, build_bm25, build_hybrid, build_hybrid_rerank,
)

retrievers = {
    'Dense': build_dense(),
    'BM25': build_bm25(),
    'Hybrid': build_hybrid(),
    'Hybrid+Rerank': build_hybrid_rerank(),
}
list(retrievers)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

['Dense', 'BM25', 'Hybrid', 'Hybrid+Rerank']

## 정성 점검 — 대표 질문에서 각 retriever의 top chunk
고유명사/숫자(엔비디아·GTC·30GW)가 들어간 질문에서 BM25/Hybrid가 dense 대비 무엇을 더/덜 잡는지 눈으로 확인.

In [3]:
sample_q = 'GTC 워싱턴에서 엔비디아가 발표한 신규 사업 영역은?'
for name, r in retrievers.items():
    print('=' * 70)
    print(name)
    for i, d in enumerate(r.invoke(sample_q)):
        snippet = d.page_content[:120].replace(chr(10), ' ')
        print(f'  [{i}] ({d.metadata.get("published_at","?")}) {snippet}...')

Dense


  [0] (2025-11-03 00:00:00) . 그래서 고객사들하고 관계도 잘 맺고 한 것 같은데 끝판왕은 엔비디아가 수혜를 벗겨 봤죠. 당연히 우리나라 기업들도 수혜를 받겠지만 엔비디아는 당장 돈을 번 거니까 매출이 잡히니까 큰 돈 벌고 간 것 같습니다. 바...
  [1] (2025-11-03 00:00:00) . 지금 젠슨항도 대만 TSMC 관계사도 만나러 가고 지금 영국도 AI 밀어줘야 돼가지고 젠슨항 지금 영국 간다고 하더라고요. 지금 바쁩니다. 진짜 한국도 와서 한 번 또 립서비스 해주면서 한국 최고 얘기해주고 또 ...
  [2] (2026-04-28 00:00:00) 등판해서 지금 엔트로픽 추격팀 특별편성에서 진두지휘하고 있다고 합니다 그러니까 지금 한편으로는 엔트로픽에 투자를 하면서도 내부적으로 위기감에 지금 재미나게 살려야 된다 큰일 난다 이렇게 하고 있는 겁니다 그러니까 개...
  [3] (2025-11-03 00:00:00) . 그래서 요것도 엔비디아 최첨단 반도체를 중국에 주냐 마냐가지고 엔비디아 추가 상승 여부도 좀 결정될 수 있을 것 같고 그래서 제가 쓸 것 같습니다. 중국이랑 1년 동안 화해하는 모드로 갈 것 같아가지고 그래서 엔...
BM25
  [0] (2025-11-03 00:00:00) . 그래서 고객사들하고 관계도 잘 맺고 한 것 같은데 끝판왕은 엔비디아가 수혜를 벗겨 봤죠. 당연히 우리나라 기업들도 수혜를 받겠지만 엔비디아는 당장 돈을 번 거니까 매출이 잡히니까 큰 돈 벌고 간 것 같습니다. 바...
  [1] (2025-11-03 00:00:00) . 지금 젠슨항도 대만 TSMC 관계사도 만나러 가고 지금 영국도 AI 밀어줘야 돼가지고 젠슨항 지금 영국 간다고 하더라고요. 지금 바쁩니다. 진짜 한국도 와서 한 번 또 립서비스 해주면서 한국 최고 얘기해주고 또 ...
  [2] (2025-11-03 00:00:00) . 달 탐사급으로 중요하게 미국 정부에서 밀어주고 있고 그 핵심을 지금 엔비디아가 갖고 있다 이렇게

  [0] (2025-11-03 00:00:00) . 그래서 고객사들하고 관계도 잘 맺고 한 것 같은데 끝판왕은 엔비디아가 수혜를 벗겨 봤죠. 당연히 우리나라 기업들도 수혜를 받겠지만 엔비디아는 당장 돈을 번 거니까 매출이 잡히니까 큰 돈 벌고 간 것 같습니다. 바...
  [1] (2025-11-03 00:00:00) . 지금 젠슨항도 대만 TSMC 관계사도 만나러 가고 지금 영국도 AI 밀어줘야 돼가지고 젠슨항 지금 영국 간다고 하더라고요. 지금 바쁩니다. 진짜 한국도 와서 한 번 또 립서비스 해주면서 한국 최고 얘기해주고 또 ...
  [2] (2025-11-03 00:00:00) . 달 탐사급으로 중요하게 미국 정부에서 밀어주고 있고 그 핵심을 지금 엔비디아가 갖고 있다 이렇게 선언을 한 셈이고요. 그래서 다 하겠다라고 선언한 겁니다. 사업 영역 확당 선언. 양자컴도 할 거고 슈퍼컴도 할 거...
  [3] (2025-11-03 00:00:00) . 다들 생각이 다르겠지만 그러면서 이제 내가 들었는데 어떤 사람들은 약 1년만에 투자금을 2배로 늘렸다고 한다. 대단하다. 매우 이상 깊다. 잘하고 있고 축하한다. 이러면서 다시 한 번 또 어디 가든 미국 주식에 ...
Hybrid+Rerank


  [0] (2025-11-03 00:00:00) . 그래서 고객사들하고 관계도 잘 맺고 한 것 같은데 끝판왕은 엔비디아가 수혜를 벗겨 봤죠. 당연히 우리나라 기업들도 수혜를 받겠지만 엔비디아는 당장 돈을 번 거니까 매출이 잡히니까 큰 돈 벌고 간 것 같습니다. 바...
  [1] (2025-11-03 00:00:00) . 달 탐사급으로 중요하게 미국 정부에서 밀어주고 있고 그 핵심을 지금 엔비디아가 갖고 있다 이렇게 선언을 한 셈이고요. 그래서 다 하겠다라고 선언한 겁니다. 사업 영역 확당 선언. 양자컴도 할 거고 슈퍼컴도 할 거...
  [2] (2025-11-03 00:00:00) . 지금 젠슨항도 대만 TSMC 관계사도 만나러 가고 지금 영국도 AI 밀어줘야 돼가지고 젠슨항 지금 영국 간다고 하더라고요. 지금 바쁩니다. 진짜 한국도 와서 한 번 또 립서비스 해주면서 한국 최고 얘기해주고 또 ...
  [3] (2025-11-03 00:00:00) . 다들 생각이 다르겠지만 그러면서 이제 내가 들었는데 어떤 사람들은 약 1년만에 투자금을 2배로 늘렸다고 한다. 대단하다. 매우 이상 깊다. 잘하고 있고 축하한다. 이러면서 다시 한 번 또 어디 가든 미국 주식에 ...


## 1) 평균 latency 측정 (질문당 retriever.invoke 평균)

In [4]:
import time
from naive_rag.constants import WEEK4_EVAL_DATA

latency = {}
for name, r in retrievers.items():
    t0 = time.perf_counter()
    for d in WEEK4_EVAL_DATA:
        r.invoke(d['question'])
    latency[name] = (time.perf_counter() - t0) / len(WEEK4_EVAL_DATA)
latency

{'Dense': 0.42241439999997965,
 'BM25': 0.04185791249983595,
 'Hybrid': 0.08332420000006095,
 'Hybrid+Rerank': 3.7652724333005607}

## 2) RAGAS 평가 (4 retriever × 10문항 × 4지표)
retriever 주입형 `run_eval` 4회 호출 → 평균 점수 표로 결합.

In [5]:
import pandas as pd
from advanced_retrieval.eval import run_eval

rows = {}
for name, r in retrievers.items():
    df = run_eval(r).to_pandas()
    rows[name] = df.mean(numeric_only=True)
table = pd.DataFrame(rows).T
table['avg_latency_s'] = pd.Series(latency)
table = table.round(4)
table.to_csv('week5_results.csv')
table

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


...

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.


,faithfulness,answer_relevancy,llm_context_precision_with_reference,context_recall,avg_latency_s
Dense,0.9917,0.6858,0.9444,0.9250,0.4224
BM25,0.9097,0.6203,0.9639,0.8500,0.0419
Hybrid,0.9909,0.6288,0.9444,0.9167,0.0833
Hybrid+Rerank,0.9784,0.6282,0.9917,0.8917,3.7653


## 3) 우승 선정 — RAGAS 4지표 평균 최고
(해석은 아래 회고 셀 참고. 운영상 latency trade-off는 §4에서 논의.)

In [6]:
metric_cols = [c for c in table.columns if c != 'avg_latency_s']
ranking = table[metric_cols].mean(axis=1).sort_values(ascending=False)
winner_name = ranking.index[0]
print('RAGAS 4지표 평균 랭킹:')
print(ranking.round(4))
print('\nWINNER:', winner_name)

RAGAS 4지표 평균 랭킹:
Dense            0.8867
Hybrid+Rerank    0.8725
Hybrid           0.8702
BM25             0.8360
dtype: float64

WINNER: Dense


## 4) 실패 케이스 발굴 — 우승 retriever로 10문항 retrieve 덤프
retrieved chunk + 생성 답변을 출력해 명백히 부족/오검색 3건을 식별한다.

In [7]:
from advanced_retrieval.eval import build_chain_for

winner = retrievers[winner_name]
chain = build_chain_for(winner)
lines = []
for i, d in enumerate(WEEK4_EVAL_DATA):
    q = d['question']
    docs = winner.invoke(q)
    ans = chain.invoke(q)
    block = [f'### Q{i+1}: {q}']
    for j, dc in enumerate(docs):
        snip = dc.page_content[:160].replace(chr(10), ' ')
        block.append(f'  [{j}] ({dc.metadata.get("published_at","?")}) {snip}...')
    block.append(f'  ANSWER: {ans[:200]}')
    lines.append(chr(10).join(block))
dump = (chr(10) * 2).join(lines)
with open('week5_failure_dump.txt', 'w') as f:
    f.write(dump)
print(dump)

### Q1: 최근 AI 투자 전쟁에서 주목한 수혜주는 무엇인가?
  [0] (2026-04-28 00:00:00) 자, 이번 주 가장 중요한 이슈 있죠? 바로 슈퍼위크입니다. 빅테크 5개가 실적 발표합니다. 그래서 가장 최근 이슈들과 함께 또 주목할 만한 빅테크와 수예주들을 같이 정리 좀 해보도록 하겠습니다. 일단 제가 봤을 때는 이번 주 실적 발표를 예상하는 것보다 그냥 잘하고 있는 기업을 같이 ...
  [1] (2026-04-28 00:00:00) 인센티브 주고 혜택을 주는 것 같아요 풀패키지로 이렇게 막 주겠죠 혜택을 주겠죠 마이크로소프트한테서 뺏어오겠다 이렇게 선언을 했어요 전방위 다한다 얘기죠 그래서 이런식으로 이게 현실화되진 않이든 현재 빅테크발 AI인프라투자 경쟁품은 더 심해지면 심해졌지 이게 끝날 기미가 보이진 않는다 ...
  [2] (2026-04-27 00:00:00) . 그 외에도 제가 말씀드렸던 것처럼, 아마존의 자체 반도체, 구글의 자체 반도체, 퀄컴 등의 유명한 그런 테크 기업의 반도체, 대부분 ARM 설계한 ARM 기반의 반도체를 쓰기 때문에, 뒤에서 오는 또 하나의 수혜주로 볼 수 있겠죠? 그래서 저 같은 경우는 이 삼총사가 CPU 수혜주로...
  [3] (2026-04-28 00:00:00) 엔트로픽 뭐 XAI 이렇게 해서 지상에서 살고 있고 데이터센터 돌아가고 있는데 어떻게 보면 이제 양극화가 되면서 너무 극단적으로 표현한 겁니다만 지금 심각하다 얘기를 느끼는 거죠 어떻게 보면은 슬프지만 이렇게 안 되기 위해서 우리가 조금 투자를 해서라도 방어 해지가 필요할 수 있겠다 그...
  ANSWER: 최근 AI 투자 전쟁에서 주목한 수혜주는 구글과 아마존입니다. 이 두 기업은 엔트로픽에 대한 대규모 투자를 진행하며, AI 인프라 투자 경쟁에서 중요한 역할을 하고 있습니다. 특히 구글은 기존 투자 대비 10배 이상의 규모로 추가 투자를 발표했으며, 아마존도 최소 3배 이상의 금액을 추가 투자한다고 밝혔습니다. 이 외에도 엔비디아

## 5) 회고 요약

| strategy | Faithful | AnswerRel | CtxPrecision | CtxRecall | latency(s) |
|---|---|---|---|---|---|
| **Dense** | 0.992 | **0.686** | 0.944 | **0.925** | 0.42 |
| BM25 | 0.910 | 0.620 | 0.964 | 0.850 | 0.04 |
| Hybrid | 0.991 | 0.629 | 0.944 | 0.917 | 0.08 |
| Hybrid+Rerank | 0.978 | 0.628 | **0.992** | 0.892 | 3.77 |

**RAGAS 4지표 평균:** Dense 0.887 > Hybrid+Rerank 0.873 > Hybrid 0.870 > BM25 0.836

**최종 선택: Dense** (6주차 baseline). 작고 깨끗한 한국어 코퍼스 + bge-m3에선 dense가 이미 품질 상한 근접. Hybrid+Rerank는 ContextPrecision만 우승(0.992)하나 latency ~9배(3.77s). 여전한 실패 3케이스(도입부 노이즈·의미 표류·broad query 발산)와 전체 분석은 [`docs/week5_retrieval_ablation.md`](../docs/week5_retrieval_ablation.md) 참조.